In [3]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
df = pd.read_csv('../data/diabetes_prediction_dataset.csv')
numeric = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorial = ['gender', 'hypertension', 'heart_disease', 'smoking_history']
multi_category= ["gender", "smoking_history"]
RANDOM_STATE = 42
TARGET_COL = "diabetes"

Разделим все данные на train и test. Подбор методов обработки признаков, настройку гиперпараметров и оценку качества будем проводить только на кросс-валидации на train. Test служит для финальной оценки качества лучшей модели.

In [5]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)


In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Основной метрикой качества выбрана Recall, поскольку в задаче прогнозирования диабета критически важно минимизировать количество ложноотрицательных классификаций (False Negative), то есть не пропустить пациентов, действительно имеющих заболевание. При этом дополнительно анализируются метрики Precision, F1-score и ROC-AUC для комплексной оценки качества модели.

## 1. Decision Tree

In [11]:
processor = ColumnTransformer(
        transformers=[
            ("category", OrdinalEncoder(), multi_category)
        ],
        remainder="passthrough",
        verbose_feature_names_out=False
    )

decision_tree = Pipeline([
    ("preprocess", processor),
    ("dectree", DecisionTreeClassifier(
        random_state=RANDOM_STATE))
])

param_grid = {
    'dectree__max_depth': [3, 5, 7, None],
    'dectree__min_samples_split': [2, 5, 10, 15],
    'dectree__min_samples_leaf': [5, 10, 15, 20],
    'dectree__criterion': ['gini', 'entropy'],
    'dectree__class_weight': [None, 'balanced']
}

gs_dt= GridSearchCV(
    decision_tree,
    param_grid=param_grid,
    cv=skf,
    scoring="recall",
    n_jobs=-1
)

gs_dt.fit(X_train, y_train)
print("Дерево решений:")
print("Лучшие параметры:", gs_dt.best_params_)
print("Лучший recall на кросс-валидации:", gs_dt.best_score_)

Дерево решений:
Лучшие параметры: {'dectree__class_weight': 'balanced', 'dectree__criterion': 'entropy', 'dectree__max_depth': 5, 'dectree__min_samples_leaf': 5, 'dectree__min_samples_split': 2}
Лучший recall на кросс-валидации: 0.9675


In [12]:
y_pred_dt = cross_val_predict(
    gs_dt.best_estimator_,
    X_train, y_train,
    cv=skf,
    n_jobs=-1
)

print("Отчёт по классам (модель decision tree):")
print(classification_report(y_train, y_pred_dt))

Отчёт по классам (модель decision tree):
              precision    recall  f1-score   support

           0       1.00      0.80      0.88     73200
           1       0.31      0.97      0.46      6800

    accuracy                           0.81     80000
   macro avg       0.65      0.88      0.67     80000
weighted avg       0.94      0.81      0.85     80000



Модель дерева решений достигла высокой полноты (Recall = 0.97) для класса пациентов с диабетом, что означает обнаружение 97% всех заболевших и минимальное количество ложноотрицательных прогнозов. Однако значение Precision = 0.31 свидетельствует о большом числе ложноположительных классификаций: значительная часть пациентов, отнесённых моделью к группе риска, фактически не имеет диабета. Следовательно, модель хорошо подходит для задач первичного скрининга, где приоритетом является минимизация пропуска больных.